# Transmission time delay

This example models a shipment of hydrogen from location A to location B. The shipment leaves A in time step 0 and arrives at B after its configured transit delay.

The model uses a closed horizon: no commodity is initially in transit, and a dispatch is only allowed when it can arrive before the final time step. Positive time delays cannot currently be combined with time-series aggregation.

## 1. Import packages

FINE defines the energy system model and its components; pandas is used to create the time series for supply and demand.

In [14]:
import pandas as pd

import fine as fn

## 2. Create the energy system model

There are two locations and four one-hour time steps. The short horizon makes the transit delay easy to see. Change `time_delay` below and rerun the notebook to experiment with a different transit time.

In [15]:
time_delay = 2
shipment_amount = 10

esM = fn.EnergySystemModel(
    locations={"A", "B"},
    commodities={"hydrogen"},
    commodityUnitsDict={"hydrogen": "kg"},
    numberOfTimeSteps=4,
    hoursPerTimeStep=1,
    costUnit="EUR",
    lengthUnit="km",
)

## 3. Add supply and demand

Ten kilograms are available at A only in time step 0. The same amount is required at B exactly `time_delay` time steps later. The demand time series is created below so it stays aligned when you change `time_delay`.

In [16]:
demand_at_b = [0] * esM.numberOfTimeSteps
demand_at_b[time_delay] = shipment_amount

esM.add(
    fn.Source(
        esM=esM,
        name="Hydrogen supply at A",
        commodity="hydrogen",
        hasCapacityVariable=False,
        operationRateMax=pd.DataFrame(
            {"A": [shipment_amount, 0, 0, 0], "B": [0, 0, 0, 0]}
        ),
    )
)

esM.add(
    fn.Sink(
        esM=esM,
        name="Hydrogen demand at B",
        commodity="hydrogen",
        hasCapacityVariable=False,
        operationRateFix=pd.DataFrame({"A": [0, 0, 0, 0], "B": demand_at_b}),
    )
)

## 4. Add a delayed transmission component

`timeDelay=time_delay` is measured in model time steps. The operation variable remains the dispatched amount at the origin; FINE shifts its commodity-balance contribution at B by that many time steps.

In [17]:
esM.add(
    fn.Transmission(
        esM=esM,
        name="Ship from A to B",
        commodity="hydrogen",
        timeDelay=time_delay,
        hasCapacityVariable=False,
        locationalEligibility=pd.DataFrame(
            [[0, 1], [1, 0]], index=["A", "B"], columns=["A", "B"]
        ),
    )
)

## 5. Optimize the model

FINE selects an available solver when none is specified. Pass `solver="glpk"`, `solver="highs"`, or another installed solver explicitly if needed.

In [18]:
esM.optimize()

Set parameter Threads to value 3
Set parameter QCPDual to value 1
Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (linux64 - "Rocky Linux 9.6 (Blue Onyx)")

CPU model: QEMU Virtual CPU version 2.5+, instruction set [SSE2]
Thread count: 8 physical cores, 8 logical processors, using up to 3 threads

Optimize a model with 12 rows, 16 columns and 24 nonzeros
Model fingerprint: 0x4d63ab15
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [0e+00, 0e+00]
  Bounds range     [1e+01, 1e+01]
  RHS range        [0e+00, 0e+00]
Presolve removed 0 rows and 8 columns
Presolve time: 0.01s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Infeasible model
model.name="unknown";
    - termination condition: infeasible
    - message from solver: <undefined>


## 6. Inspect the delayed arrival

The table and chart make the shift explicit: dispatch leaves A at time step 0, while the matching arrival is placed at B `time_delay` steps later. In this example there are no transmission losses, so the arrival and dispatch quantities are equal.

In [ ]:
transmission_model = esM.componentModelingDict["TransmissionModel"]
ip_name = esM.investmentPeriodNames[0]
flows = transmission_model.getOptimalValues(
    "operationVariablesOptimum", ip=ip_name
)["values"]

dispatch = flows.loc[[("Ship from A to B", "A", "B")]].T.iloc[:, 0]
arrival = dispatch.shift(time_delay, fill_value=0)
demand = pd.Series(demand_at_b, index=dispatch.index, name="demand at B")

timeline = pd.concat(
    [
        dispatch.rename("dispatch from A"),
        arrival.rename("arrival at B"),
        demand,
    ],
    axis=1,
)
timeline.index.name = "time step"
display(timeline)
timeline.plot.bar(rot=0, ylabel="hydrogen (kg per time step)", figsize=(8, 4));